# 06 · Uso de jugadores

**Pregunta central 2 del Plan Maestro (ideas tácticas del entrenador):** ¿qué roles asigna Jardine por posición, qué tan fija es su alineación titular, y qué efecto tienen las sustituciones sobre el rendimiento del equipo?

Tres bloques:
1. **Alineaciones titulares** — formación más usada, continuidad del once entre partidos consecutivos, jugadores indiscutibles vs. rotación.
2. **Mapas de calor individuales** — dónde actúan los jugadores clave en la cancha.
3. **Efecto de sustituciones** — comparando las métricas del equipo en los minutos antes/después de cada cambio (no contra la línea base de la temporada, como en el bloque 04, sino contra el "antes" inmediato de cada sustitución).

In [ ]:
import sys
sys.path.append("../src")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from framework_analitico import (
    normalizar_direccion_ataque,
    extraer_alineaciones_titulares,
    resumen_titulares,
    cambios_alineacion_consecutivos,
    identificar_sustituciones,
    comparar_metricas_pre_post_sustitucion,
    ubicaciones_jugador,
)

EQUIPO = "América"  # cambia esto para reaplicar el analisis a otro entrenador/equipo

events_jardine = pd.read_csv("../data/raw/events_jardine_america.csv", low_memory=False)
print(f"Eventos cargados: {len(events_jardine)}")

events_norm = normalizar_direccion_ataque(events_jardine, equipo=EQUIPO)
print(f"Partidos: {events_norm['match_id'].nunique()}")

# Metadatos de partido (match_date) para poder ordenar cronológicamente — ya quedaron
# cacheados en el bloque 05, aquí solo se reutilizan.
partidos = pd.read_csv("../data/raw/partidos_jardine_america.csv")
partidos["match_date"] = pd.to_datetime(partidos["match_date"])
print(f"Metadatos de partidos: {len(partidos)}")

## 1. Alineaciones titulares

### 1.1 Formación más usada

In [ ]:
alineaciones = extraer_alineaciones_titulares(events_norm, equipo=EQUIPO)
print(f"Partidos con alineación titular identificada: {len(alineaciones)}")
print()
print(alineaciones["formacion"].value_counts())

### 1.2 Jugadores más usados como titular

In [ ]:
titulares = resumen_titulares(alineaciones)
titulares.head(15)

### 1.3 Continuidad de la alineación entre partidos consecutivos

Se ordena por fecha real de partido (metadatos del bloque 05) antes de medir continuidad — "consecutivo" significa el siguiente partido jugado, no el siguiente match_id.

In [ ]:
alineaciones_ordenadas = alineaciones.merge(
    partidos[["match_id", "match_date"]], on="match_id", how="left"
).sort_values("match_date").reset_index(drop=True)

continuidad = cambios_alineacion_consecutivos(alineaciones_ordenadas)

print(f"Cambios promedio en el once entre partidos consecutivos: {continuidad['cambios'].mean():.2f} de 11")
print(f"Partidos con el once IDÉNTICO al anterior: {(continuidad['cambios'] == 0).sum()} de {len(continuidad)}")
print()
continuidad["cambios"].value_counts().sort_index()

## 2. Mapas de calor individuales

Ubicación de los eventos (todos los tipos con coordenadas) de los jugadores más usados como titular, para ver el rol que cumple cada uno en la cancha.

In [ ]:
jugadores_clave = titulares.head(6)["jugador"].tolist()

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, jugador in zip(axes.flat, jugadores_clave):
    ubicaciones = ubicaciones_jugador(events_norm, equipo=EQUIPO, jugador=jugador)
    ax.hist2d(ubicaciones["x"], ubicaciones["y"], bins=[24, 16], range=[[0, 120], [0, 80]], cmap="YlOrRd")
    ax.set_title(jugador, fontsize=9)
    ax.set_xlim(0, 120)
    ax.set_ylim(0, 80)
    ax.invert_yaxis()
    ax.set_xticks([])
    ax.set_yticks([])

plt.suptitle(f"Mapas de calor — jugadores más usados como titular ({EQUIPO})")
plt.tight_layout()
plt.savefig("../outputs/figures/mapas_calor_jugadores.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Efecto de las sustituciones

Para cada cambio, se comparan las métricas del equipo en los 5 minutos ANTES del cambio contra los 5 minutos DESPUÉS (no contra la línea base de la temporada, como en el bloque 04 — aquí el punto de comparación es el "antes" inmediato de ese mismo momento del partido).

In [ ]:
sustituciones = identificar_sustituciones(events_norm, equipo=EQUIPO)
print(f"Total de sustituciones: {len(sustituciones)}")
sustituciones.head(10)

In [ ]:
efecto_cambios = comparar_metricas_pre_post_sustitucion(
    events_norm, equipo=EQUIPO, sustituciones=sustituciones, ventana_segundos=300
)

columnas_diff = [c for c in efecto_cambios.columns if c.startswith("diff_")]
print("Promedio del efecto de TODAS las sustituciones (después - antes):")
efecto_cambios[columnas_diff].mean().round(3)

### 3.1 ¿El efecto depende del minuto del cambio?

Se agrupan las sustituciones en dos franjas (antes/después del minuto 70, un corte habitual entre "cambios tácticos" y "cambios de cierre de partido") para ver si el patrón cambia.

In [ ]:
efecto_cambios["franja"] = np.where(efecto_cambios["minute"] < 70, "Antes del min 70", "Min 70 o después")

resumen_por_franja = efecto_cambios.groupby("franja")[columnas_diff].mean().round(3)
resumen_por_franja["n_cambios"] = efecto_cambios.groupby("franja").size()
resumen_por_franja

## 4. Síntesis: ¿qué rol asigna Jardine y cómo usa el banco?

**1. Formación — base clara, pero no rígida.** El 4-2-3-1 es el sistema más usado (57/129 partidos, 44%), seguido del 4-4-2 (26, 20%) y el 3-4-3 (14, 11%). En total se usaron 9 formaciones distintas, alternando entre líneas de 3 y de 4 según el partido — no hay un único dibujo fijo.

**2. Alineación titular — columna vertebral fija + rotación en el resto.** Cuatro jugadores son casi indiscutibles: Malagón (portero, 109/129 = 84%), Fidalgo (91), Israel Reyes (90) y Zendejas (90). En otras posiciones sí hay rotación real (ej. lateral izquierdo repartido entre Borja 55 y Calderón 40; el 9 repartido entre Henry Martín 61 y otros). En promedio cambian **3.57 de 11 titulares** entre partidos consecutivos, y solo el 6% de los partidos repite el once exacto del anterior — hay rotación constante, pero moderada (72% de los partidos mantiene al menos 7 de 11).

**3. Mapas de calor — los roles de banda son más disciplinados espacialmente.** El lateral (Kevin Álvarez) se concentra en una franja angosta pegada a una banda, mientras el mediocampista de contención (Jonathan dos Santos) y el extremo (Zendejas) cubren zonas más amplias y centrales — confirma que los laterales tienen una función más especializada/fija que los jugadores de banda ofensiva o del medio.

**4. Sustituciones — dos tipos de cambio con efectos casi opuestos.** El promedio de TODAS las sustituciones (603 en total, ~4.7 por partido) sale plano porque mezcla dos patrones que se cancelan entre sí:
- **Antes del min 70 (260 cambios) = ajuste táctico ofensivo:** sube la línea (+3.43), gana posesión (+5.37pp) y juega más en corto (-3.53pp pases largos) tras el cambio.
- **Min 70 en adelante (343 cambios) = gestión de cierre:** baja la línea (-2.70, casi espejo), pierde algo de posesión (-1.34pp) y juega más directo (+0.95pp pases largos).
- La única métrica que sube en ambos casos es la presión (+0.126 antes del 70, +0.230 después) — las piernas frescas presionan más intenso incluso cuando el plan general es replegarse.

**Lectura general para el Plan Maestro:** Jardine tiene una idea de juego clara sostenida por un núcleo fijo de 4 jugadores, pero gestiona el resto del plantel con rotación activa y bastante variedad táctica (9 formaciones). El uso del banco es deliberado y bimodal: los cambios tempranos buscan subir el nivel del equipo (más posesión, línea más alta), mientras los tardíos priorizan cerrar el partido (línea más baja, juego más directo) — un patrón de sustituciones consistente con un entrenador que lee el partido y ajusta el plan según el momento, no solo por desgaste físico.